In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import jax.numpy as jnp
import jax.scipy as jsp
from jax.experimental import sparse
import jax

data = sparse.BCOO.fromdense(jnp.identity(4))
# sparse.sparsify(jsp.linalg.expm)(data)

In [ ]:
# Single-cell, JAX-jit-safe sparse BCOO expm approximation (Taylor on a fixed sparsity pattern)
# - Never densifies
# - No boolean indexing inside jit
# - No dynamic nnz inside jit
# - Works by precomputing a fixed pattern P (union of A^k sparsity up to 'order') on host,
#   then projecting all intermediates onto P inside jit.

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import sparse


def pattern_union_upto_k(indices_np: np.ndarray, n: int, kmax: int) -> np.ndarray:
    """Host-side: union of sparsity patterns of A^k for k=0..kmax via graph reachability."""
    if indices_np.size == 0:
        return np.array([(i, i) for i in range(n)], dtype=np.int32)

    rows = indices_np[:, 0].tolist()
    cols = indices_np[:, 1].tolist()

    adj = {i: set() for i in range(n)}
    for r, c in zip(rows, cols):
        adj[r].add(c)

    P = set((i, i) for i in range(n))  # include identity pattern (k=0)

    for i in range(n):
        frontier = set(adj[i])  # depth 1 reachable
        for _depth in range(1, kmax + 1):
            for j in frontier:
                P.add((i, j))
            nxt = set()
            for u in frontier:
                nxt |= adj.get(u, set())
            frontier = nxt
            if not frontier:
                break

    return np.array(sorted(P), dtype=np.int32)


def make_projector(P_indices: jnp.ndarray, n: int):
    """Returns project(B: BCOO) -> data aligned to fixed P_indices, JIT-safe."""
    P_keys = (P_indices[:, 0] * n + P_indices[:, 1]).astype(jnp.int32)
    m = P_keys.shape[0]

    def project(B: sparse.BCOO):
        keys = (B.indices[:, 0] * n + B.indices[:, 1]).astype(jnp.int32)

        pos = jnp.searchsorted(P_keys, keys)
        pos_clip = jnp.clip(pos, 0, m - 1)

        ok = (pos < m) & (P_keys[pos_clip] == keys)
        ok_f = ok.astype(B.data.dtype)  # 0/1 mask, avoids boolean indexing

        dataP = jnp.zeros((m,), dtype=B.data.dtype)
        # Scatter-add all entries; invalid ones contribute 0
        dataP = dataP.at[pos_clip].add(B.data * ok_f)
        return dataP

    return project


def sparse_expm_bcoo_fixed_pattern(A: sparse.BCOO, order: int = 20, prune_tol: float = 0.0) -> sparse.BCOO:
    """
    Approximate expm(A) via truncated Taylor series on a fixed sparsity pattern P.
    exp(A) ≈ Σ_{k=0..order} A^k / k!

    NOTE: For generic sparse A, exp(A) is dense in general. This stays sparse by restricting to P
    (union of patterns of A^k up to 'order') and optionally zeroing small coefficients in data.
    """
    if A.shape[0] != A.shape[1]:
        raise ValueError("A must be square.")
    n = int(A.shape[0])

    # Host-side fixed pattern P (depends only on indices)
    P_np = pattern_union_upto_k(np.array(A.indices), n, int(order))
    P_indices = jnp.array(P_np)

    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    I = sparse.eye(n, dtype=A.dtype)
    termP0 = project(I)
    resultP0 = termP0

    @jax.jit
    def run(termP, resultP):
        def body(k, state):
            termP, resultP = state
            term = as_P(termP)

            # term <- A @ term / k  (all sparse)
            prod = sparse.bcoo_dot_general(
                A, term,
                dimension_numbers=(((1,), (0,)), ((), ()))
            )
            new_termP = project(prod) / k

            if prune_tol != 0.0:
                new_termP = jnp.where(jnp.abs(new_termP) > prune_tol, new_termP, 0.0)

            new_resultP = resultP + new_termP
            if prune_tol != 0.0:
                new_resultP = jnp.where(jnp.abs(new_resultP) > prune_tol, new_resultP, 0.0)

            return (new_termP, new_resultP)

        termP, resultP = jax.lax.fori_loop(1, order + 1, body, (termP, resultP))
        return resultP

    resultP = run(termP0, resultP0)
    return sparse.BCOO((resultP, P_indices), shape=(n, n))


# --- Example ---
A = sparse.BCOO.fromdense(jnp.array([
    [0., 1., 0., 0.],
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
    [0., 0., 0., 0.],
], dtype=jnp.float32))

E = sparse_expm_bcoo_fixed_pattern(A, order=30, prune_tol=1e-12)
print("stored nnz:", E.nse)
# print(E.todense())  # only for sanity check if you allow dense visualization

E.todense() - jsp.linalg.expm(A.todense())


stored nnz: 10


Array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)

In [ ]:
# Sparse BCOO expm with Scaling+Squaring (no dense at any point)
# - Uses truncated Taylor for exp(A / 2^s) on a fixed pattern
# - Then squares s times, each squaring done on its own fixed pattern
# - No boolean indexing inside jit, no dynamic nnz inside jit
# - Never calls todense / never materializes (n,n) arrays

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import sparse


# -----------------------------
# Host-side pattern utilities
# -----------------------------
def pattern_union_upto_k(indices_np: np.ndarray, n: int, kmax: int) -> np.ndarray:
    """Union of sparsity patterns of A^k for k=0..kmax via reachability (host-side)."""
    P = set((i, i) for i in range(n))  # identity pattern
    if indices_np.size == 0 or kmax <= 0:
        return np.array(sorted(P), dtype=np.int32)

    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    for i in range(n):
        frontier = set(adj[i])  # depth 1
        for _depth in range(1, kmax + 1):
            for j in frontier:
                P.add((i, j))
            nxt = set()
            for u in frontier:
                nxt |= adj.get(u, set())
            frontier = nxt
            if not frontier:
                break

    return np.array(sorted(P), dtype=np.int32)


def pattern_square(indices_np: np.ndarray, n: int) -> np.ndarray:
    """Sparsity pattern of M@M given pattern(M) (host-side)."""
    if indices_np.size == 0:
        return np.array([(i, i) for i in range(n)], dtype=np.int32)

    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    P2 = set()
    for i in range(n):
        for k in adj[i]:
            for j in adj.get(k, ()):
                P2.add((i, j))

    # It's often useful to keep diagonal reachable too
    for i in range(n):
        P2.add((i, i))

    return np.array(sorted(P2), dtype=np.int32)


# -----------------------------
# JIT-safe projector (no bool indexing)
# -----------------------------
def make_projector(P_indices: jnp.ndarray, n: int):
    """
    Returns project(BCOO) -> data aligned to fixed P_indices.
    JIT-safe: uses masked scatter-add, no boolean indexing.
    """
    P_keys = (P_indices[:, 0] * n + P_indices[:, 1]).astype(jnp.int32)
    m = P_keys.shape[0]

    def project(B: sparse.BCOO):
        keys = (B.indices[:, 0] * n + B.indices[:, 1]).astype(jnp.int32)

        pos = jnp.searchsorted(P_keys, keys)
        pos_clip = jnp.clip(pos, 0, m - 1)

        ok = (pos < m) & (P_keys[pos_clip] == keys)
        ok_f = ok.astype(B.data.dtype)

        dataP = jnp.zeros((m,), dtype=B.data.dtype)
        dataP = dataP.at[pos_clip].add(B.data * ok_f)  # duplicates sum automatically
        return dataP

    return project


# -----------------------------
# Sparse norm estimate (no dense)
# -----------------------------
def max_abs_row_sum(A: sparse.BCOO) -> jnp.ndarray:
    """Cheap ||A||_∞ estimate: max_i sum_j |A_ij|, computed from BCOO."""
    n = A.shape[0]
    rows = A.indices[:, 0]
    row_sums = jnp.zeros((n,), dtype=A.data.dtype).at[rows].add(jnp.abs(A.data))
    return jnp.max(row_sums)


# -----------------------------
# Taylor expm on fixed pattern (JIT)
# -----------------------------
def taylor_expm_fixed_pattern(A: sparse.BCOO, order: int, P_indices: jnp.ndarray, prune_tol: float):
    n = int(A.shape[0])
    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    I = sparse.eye(n, dtype=A.dtype)
    termP0 = project(I)
    resultP0 = termP0

    @jax.jit
    def run(termP, resultP):
        def body(k, state):
            termP, resultP = state
            term = as_P(termP)

            prod = sparse.bcoo_dot_general(
                A, term,
                dimension_numbers=(((1,), (0,)), ((), ()))
            )
            new_termP = project(prod) / k

            if prune_tol != 0.0:
                new_termP = jnp.where(jnp.abs(new_termP) > prune_tol, new_termP, 0.0)

            new_resultP = resultP + new_termP
            if prune_tol != 0.0:
                new_resultP = jnp.where(jnp.abs(new_resultP) > prune_tol, new_resultP, 0.0)

            return (new_termP, new_resultP)

        termP, resultP = jax.lax.fori_loop(1, order + 1, body, (termP, resultP))
        return resultP

    resultP = run(termP0, resultP0)
    return sparse.BCOO((resultP, P_indices), shape=(n, n))


# -----------------------------
# One squaring step on fixed pattern (JIT)
# -----------------------------
def square_fixed_pattern(M: sparse.BCOO, P_indices: jnp.ndarray, prune_tol: float):
    n = int(M.shape[0])
    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    # represent M on P
    MP = project(M)

    @jax.jit
    def run(MP):
        X = as_P(MP)
        prod = sparse.bcoo_dot_general(
            X, X,
            dimension_numbers=(((1,), (0,)), ((), ()))
        )
        outP = project(prod)
        if prune_tol != 0.0:
            outP = jnp.where(jnp.abs(outP) > prune_tol, outP, 0.0)
        return outP

    outP = run(MP)
    return sparse.BCOO((outP, P_indices), shape=(n, n))


# -----------------------------
# Scaling + squaring sparse expm (no dense)
# -----------------------------
def sparse_expm_bcoo_scaling_squaring(
    A: sparse.BCOO,
    order: int = 12,          # Taylor order for exp(A/2^s)
    prune_tol: float = 0.0,   # optional coefficient thresholding on fixed patterns
    theta: float = 0.5,       # target norm for scaled operator (smaller => more stable, more squarings)
):
    if A.shape[0] != A.shape[1]:
        raise ValueError("A must be square.")
    n = int(A.shape[0])

    # Choose scaling s so that ||A/2^s||_∞ ~<= theta (roughly)
    norm_est = max_abs_row_sum(A)
    s = jnp.maximum(0, jnp.ceil(jnp.log2(norm_est / theta)).astype(jnp.int32))

    # Scale A -> A_scaled = A / 2^s (scale only data; indices unchanged)
    scale = jnp.power(2.0, -s).astype(A.data.dtype)
    A_scaled = sparse.BCOO((A.data * scale, A.indices), shape=A.shape)

    # Fixed pattern for Taylor phase: union pattern(A_scaled^k), k=0..order
    P0_np = pattern_union_upto_k(np.array(A_scaled.indices), n, int(order))
    P0 = jnp.array(P0_np)
    E = taylor_expm_fixed_pattern(A_scaled, order=order, P_indices=P0, prune_tol=prune_tol)

    # Squaring phase: repeat s times; each step builds a new fixed pattern for that squaring
    s_int = int(s)  # materialize as python int for the host-side loop
    for _ in range(s_int):
        P_sq_np = pattern_square(np.array(E.indices), n)
        P_sq = jnp.array(P_sq_np)
        E = square_fixed_pattern(E, P_indices=P_sq, prune_tol=prune_tol)

    return E


# -----------------------------
# Example
# -----------------------------
A = sparse.BCOO.fromdense(jnp.array([
    [0., 1., 0., 0.],
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
    [0., 0., 0., 0.],
], dtype=jnp.float32))

E = sparse_expm_bcoo_scaling_squaring(A, order=12, prune_tol=1e-12, theta=0.5)
print("stored nnz:", E.nse)
# print(E.todense())  # only if you allow dense visualization

E.todense() - jsp.linalg.expm(A.todense())


stored nnz: 10


In [4]:
# Sparse BCOO expm with Scaling+Squaring (REAL + COMPLEX)
# - No dense matrices ever
# - JIT-safe (static nnz inside jit)
# - Works for float or complex BCOO
# - Uses Taylor on a fixed pattern + sparse squaring

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import sparse


# -----------------------------
# Host-side sparsity patterns
# -----------------------------
def pattern_union_upto_k(indices_np: np.ndarray, n: int, kmax: int) -> np.ndarray:
    P = set((i, i) for i in range(n))
    if indices_np.size == 0 or kmax <= 0:
        return np.array(sorted(P), dtype=np.int32)

    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    for i in range(n):
        frontier = set(adj[i])
        for _ in range(kmax):
            for j in frontier:
                P.add((i, j))
            nxt = set()
            for u in frontier:
                nxt |= adj.get(u, set())
            frontier = nxt
            if not frontier:
                break

    return np.array(sorted(P), dtype=np.int32)


def pattern_square(indices_np: np.ndarray, n: int) -> np.ndarray:
    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    P2 = set((i, i) for i in range(n))
    for i in range(n):
        for k in adj[i]:
            for j in adj.get(k, ()):
                P2.add((i, j))

    return np.array(sorted(P2), dtype=np.int32)


# -----------------------------
# JIT-safe projector
# -----------------------------
def make_projector(P_indices: jnp.ndarray, n: int):
    P_keys = (P_indices[:, 0] * n + P_indices[:, 1]).astype(jnp.int32)
    m = P_keys.shape[0]

    def project(B: sparse.BCOO):
        keys = (B.indices[:, 0] * n + B.indices[:, 1]).astype(jnp.int32)
        pos = jnp.searchsorted(P_keys, keys)
        pos_clip = jnp.clip(pos, 0, m - 1)

        ok = (pos < m) & (P_keys[pos_clip] == keys)
        ok_f = ok.astype(B.data.dtype)   # works for complex

        dataP = jnp.zeros((m,), dtype=B.data.dtype)
        dataP = dataP.at[pos_clip].add(B.data * ok_f)
        return dataP

    return project


# -----------------------------
# Sparse norm estimate (complex-safe)
# -----------------------------
def max_abs_row_sum(A: sparse.BCOO):
    n = A.shape[0]
    rows = A.indices[:, 0]
    row_sums = jnp.zeros((n,), dtype=A.data.real.dtype)
    row_sums = row_sums.at[rows].add(jnp.abs(A.data))
    return jnp.max(row_sums)


# -----------------------------
# Taylor expm on fixed pattern
# -----------------------------
def taylor_expm_fixed_pattern(A, order, P_indices, prune_tol):
    n = int(A.shape[0])
    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    I = sparse.eye(n, dtype=A.data.dtype)
    termP0 = project(I)
    resultP0 = termP0

    @jax.jit
    def run(termP, resultP):
        def body(k, state):
            termP, resultP = state
            term = as_P(termP)

            prod = sparse.bcoo_dot_general(
                A, term,
                dimension_numbers=(((1,), (0,)), ((), ()))
            )
            new_termP = project(prod) / jnp.asarray(k, dtype=A.data.dtype)

            if prune_tol != 0.0:
                new_termP = jnp.where(jnp.abs(new_termP) > prune_tol, new_termP, 0)

            new_resultP = resultP + new_termP
            if prune_tol != 0.0:
                new_resultP = jnp.where(jnp.abs(new_resultP) > prune_tol, new_resultP, 0)

            return new_termP, new_resultP

        return jax.lax.fori_loop(1, order + 1, body, (termP, resultP))[1]

    return sparse.BCOO((run(termP0, resultP0), P_indices), shape=(n, n))


# -----------------------------
# Squaring on fixed pattern
# -----------------------------
def square_fixed_pattern(M, P_indices, prune_tol):
    n = int(M.shape[0])
    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    MP = project(M)

    @jax.jit
    def run(MP):
        X = as_P(MP)
        prod = sparse.bcoo_dot_general(
            X, X,
            dimension_numbers=(((1,), (0,)), ((), ()))
        )
        outP = project(prod)
        if prune_tol != 0.0:
            outP = jnp.where(jnp.abs(outP) > prune_tol, outP, 0)
        return outP

    return sparse.BCOO((run(MP), P_indices), shape=(n, n))


# -----------------------------
# Scaling + squaring sparse expm (COMPLEX-SAFE)
# -----------------------------
def sparse_expm_bcoo_scaling_squaring(
    A: sparse.BCOO,
    order: int = 10,
    prune_tol: float = 0.0,
    theta: float = 0.5,
):
    n = int(A.shape[0])
    norm_est = max_abs_row_sum(A)

    s = int(jnp.maximum(0, jnp.ceil(jnp.log2(norm_est / theta))))
    scale = jnp.asarray(2.0 ** (-s), dtype=A.data.dtype)

    A_scaled = sparse.BCOO((A.data * scale, A.indices), shape=A.shape)

    P0 = jnp.array(pattern_union_upto_k(np.array(A.indices), n, order))
    E = taylor_expm_fixed_pattern(A_scaled, order, P0, prune_tol)

    for _ in range(s):
        P_sq = jnp.array(pattern_square(np.array(E.indices), n))
        E = square_fixed_pattern(E, P_sq, prune_tol)

    return E


# -----------------------------
# Example: COMPLEX matrix
# -----------------------------
A = sparse.BCOO.fromdense(jnp.array([
    [0.+0.j, 1.+1.j, 0.+0.j],
    [0.+0.j, 0.+0.j, 2.-1.j],
    [0.+0.j, 0.+0.j, 0.+0.j],
], dtype=jnp.complex64))

E = sparse_expm_bcoo_scaling_squaring(A, order=10, prune_tol=1e-12)
print("stored nnz:", E.nse)
# print(E.todense())  # optional sanity check

E.todense() - jsp.linalg.expm(A.todense())


stored nnz: 6


Array([[0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j]], dtype=complex64)

In [6]:
import jaxquantum as jqt
A = jqt.create(10).to_sparse_bcoo().data
E = sparse_expm_bcoo_scaling_squaring(A, order=12, prune_tol=1e-12, theta=0.5)

jnp.max(jnp.abs(E.todense() - jsp.linalg.expm(A.todense())))

Array(8.8817842e-16, dtype=float64)

In [10]:
%timeit sparse_expm_bcoo_scaling_squaring(A, order=12, prune_tol=1e-12, theta=0.5)

1.87 s ± 49.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%timeit jsp.linalg.expm(A.todense())

874 µs ± 15 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
# Efficient sparse BCOO expm with Scaling + Squaring (REAL + COMPLEX)
# - Never densifies
# - JIT-safe (no dynamic nnz, no boolean indexing in jit)
# - Precomputes all sparsity patterns on host (no device->host index roundtrips)
# - Uses Taylor on fixed pattern P0 for exp(A/2^s), then s squarings on P1..Ps
#
# Caveat (math): for generic sparse A, exp(A) is dense in general; this stays sparse by
# restricting to a fixed pattern chain derived from A's sparsity graph.

import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import sparse


# -----------------------------
# Host-side sparsity patterns
# -----------------------------
def pattern_union_upto_k(indices_np: np.ndarray, n: int, kmax: int) -> np.ndarray:
    """Union pattern of A^k for k=0..kmax via reachability (host-side)."""
    P = set((i, i) for i in range(n))
    if indices_np.size == 0 or kmax <= 0:
        return np.array(sorted(P), dtype=np.int32)

    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    for i in range(n):
        frontier = set(adj[i])
        for _ in range(kmax):
            for j in frontier:
                P.add((i, j))
            nxt = set()
            for u in frontier:
                nxt |= adj.get(u, set())
            frontier = nxt
            if not frontier:
                break

    return np.array(sorted(P), dtype=np.int32)


def pattern_square(indices_np: np.ndarray, n: int) -> np.ndarray:
    """Pattern of M@M given pattern(M) (host-side)."""
    adj = {i: set() for i in range(n)}
    for r, c in indices_np:
        adj[int(r)].add(int(c))

    P2 = set((i, i) for i in range(n))
    for i in range(n):
        for k in adj[i]:
            for j in adj.get(k, ()):
                P2.add((i, j))

    return np.array(sorted(P2), dtype=np.int32)


def build_pattern_chain(A_indices_np: np.ndarray, n: int, order: int, s: int) -> list[np.ndarray]:
    """
    Precompute patterns:
      P0 = union pattern(A^k), k=0..order
      P_{i+1} = pattern(P_i squared)
    """
    P_chain = [pattern_union_upto_k(A_indices_np, n, order)]
    for _ in range(s):
        P_chain.append(pattern_square(P_chain[-1], n))
    return P_chain


# -----------------------------
# JIT-safe projector (masked scatter-add; no bool indexing)
# -----------------------------
def make_projector(P_indices: jnp.ndarray, n: int):
    P_keys = (P_indices[:, 0] * n + P_indices[:, 1]).astype(jnp.int32)
    m = P_keys.shape[0]

    def project(B: sparse.BCOO):
        keys = (B.indices[:, 0] * n + B.indices[:, 1]).astype(jnp.int32)
        pos = jnp.searchsorted(P_keys, keys)
        pos_clip = jnp.clip(pos, 0, m - 1)

        ok = (pos < m) & (P_keys[pos_clip] == keys)
        ok_f = ok.astype(B.data.dtype)

        dataP = jnp.zeros((m,), dtype=B.data.dtype)
        dataP = dataP.at[pos_clip].add(B.data * ok_f)  # duplicates sum automatically
        return dataP

    return project


# -----------------------------
# Sparse norm estimate (complex-safe)
# -----------------------------
def max_abs_row_sum(A: sparse.BCOO):
    """Cheap ||A||_∞ estimate: max_i sum_j |A_ij|. Works for complex."""
    n = int(A.shape[0])
    rows = A.indices[:, 0]
    row_sums = jnp.zeros((n,), dtype=jnp.abs(A.data).dtype)
    row_sums = row_sums.at[rows].add(jnp.abs(A.data))
    return jnp.max(row_sums)


# -----------------------------
# Taylor expm on fixed pattern (JIT)
# -----------------------------
def taylor_expm_fixed_pattern(A: sparse.BCOO, order: int, P_indices: jnp.ndarray, prune_tol: float):
    n = int(A.shape[0])
    project = make_projector(P_indices, n)

    def as_P(dataP):
        return sparse.BCOO((dataP, P_indices), shape=(n, n))

    I = sparse.eye(n, dtype=A.data.dtype)
    termP0 = project(I)
    resultP0 = termP0

    inv_k = (1.0 / jnp.arange(1, order + 1, dtype=jnp.float32)).astype(A.data.dtype)

    @jax.jit
    def run(termP, resultP):
        def body(i, state):
            # i = 0..order-1 corresponds to k=i+1
            termP, resultP = state
            term = as_P(termP)

            prod = sparse.bcoo_dot_general(
                A, term,
                dimension_numbers=(((1,), (0,)), ((), ()))
            )

            new_termP = project(prod) * inv_k[i]

            if prune_tol != 0.0:
                new_termP = jnp.where(jnp.abs(new_termP) > prune_tol, new_termP, 0)

            new_resultP = resultP + new_termP
            if prune_tol != 0.0:
                new_resultP = jnp.where(jnp.abs(new_resultP) > prune_tol, new_resultP, 0)

            return (new_termP, new_resultP)

        termP, resultP = jax.lax.fori_loop(0, order, body, (termP, resultP))
        return resultP

    resultP = run(termP0, resultP0)
    return sparse.BCOO((resultP, P_indices), shape=(n, n))


# -----------------------------
# One squaring step on fixed pattern (JIT)
# -----------------------------
def square_fixed_pattern(M: sparse.BCOO, P_in: jnp.ndarray, P_out: jnp.ndarray, prune_tol: float):
    """
    Compute (M @ M) projected onto P_out.
    M is assumed representable on P_in (i.e. its indices subset of P_in), but we don't rely on it.
    """
    n = int(M.shape[0])

    project_in = make_projector(P_in, n)
    project_out = make_projector(P_out, n)

    def as_P(dataP, P):
        return sparse.BCOO((dataP, P), shape=(n, n))

    MP = project_in(M)

    @jax.jit
    def run(MP):
        X = as_P(MP, P_in)
        prod = sparse.bcoo_dot_general(
            X, X,
            dimension_numbers=(((1,), (0,)), ((), ()))
        )
        outP = project_out(prod)
        if prune_tol != 0.0:
            outP = jnp.where(jnp.abs(outP) > prune_tol, outP, 0)
        return outP

    outP = run(MP)
    return sparse.BCOO((outP, P_out), shape=(n, n))


# -----------------------------
# Scaling + squaring sparse expm (optimized)
# -----------------------------
def sparse_expm_bcoo_scaling_squaring(
    A: sparse.BCOO,
    order: int = 10,          # Taylor order for exp(A/2^s)
    prune_tol: float = 0.0,   # optional coefficient thresholding (still fixed-nnz)
    theta: float = 0.5,       # target scaled norm; smaller => more stable, more squarings
    s_override  = None,
):
    if A.shape[0] != A.shape[1]:
        raise ValueError("A must be square.")
    n = int(A.shape[0])

    # Choose scaling factor s (host int). Allow override to avoid recompiles across calls.
    if s_override is None:
        norm_est = max_abs_row_sum(A)
        s = int(jnp.maximum(0, jnp.ceil(jnp.log2(norm_est / theta))).astype(jnp.int32))
    else:
        s = int(s_override)

    # Scale A -> A_scaled = A / 2^s (sparse-only: scale data, keep indices)
    scale = jnp.asarray(2.0 ** (-s), dtype=A.data.dtype)
    A_scaled = sparse.BCOO((A.data * scale, A.indices), shape=A.shape)

    # Precompute the full pattern chain on host ONCE (no device index round-trips)
    A_idx_np = np.array(A.indices, dtype=np.int32)
    P_chain_np = build_pattern_chain(A_idx_np, n, order, s)
    P_chain = [jnp.array(P, dtype=jnp.int32) for P in P_chain_np]  # device constants

    # Taylor phase on P0
    E = taylor_expm_fixed_pattern(A_scaled, order=order, P_indices=P_chain[0], prune_tol=prune_tol)

    # Squaring phase using precomputed patterns P1..Ps
    for i in range(s):
        E = square_fixed_pattern(E, P_in=P_chain[i], P_out=P_chain[i + 1], prune_tol=prune_tol)

    return E


# -----------------------------
# Example (complex)
# -----------------------------
A = sparse.BCOO.fromdense(jnp.array([
    [0.+0.j, 1.+1.j, 0.+0.j],
    [0.+0.j, 0.+0.j, 2.-1.j],
    [0.+0.j, 0.+0.j, 0.+0.j],
], dtype=jnp.complex64))

E = sparse_expm_bcoo_scaling_squaring(A, order=10, prune_tol=1e-12, theta=0.5)
print("stored nnz:", E.nse)
# print(E.todense())  # optional sanity check


stored nnz: 6


In [13]:
import jaxquantum as jqt
A = jqt.create(10).to_sparse_bcoo().data
E = sparse_expm_bcoo_scaling_squaring(A, order=12, prune_tol=1e-12, theta=0.5)

jnp.max(jnp.abs(E.todense() - jsp.linalg.expm(A.todense())))

Array(3.92539357e-09, dtype=float64)

In [14]:
%timeit sparse_expm_bcoo_scaling_squaring(A, order=12, prune_tol=1e-12, theta=0.5)

1.88 s ± 72.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
